In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

print('Train shape:', train.shape)
print('Test shape:', test.shape)

Train shape: (1460, 81)
Test shape: (1459, 80)


In [2]:
#remove 2 suspecious outliers
train = train[~((train['GrLivArea'] > 4000) & 
                (train['SalePrice'] < 200000))]
print('Train shape after removing outliers:', train.shape)

Train shape after removing outliers: (1458, 81)


In [3]:
#separate target variable befor transformation

y_train = np.log1p(train['SalePrice'])

train = train.drop(['SalePrice', 'Id'], axis=1)
test = test.drop(['Id'], axis=1)

print('Train features shape:', train.shape)
print('Test shape:', test.shape)
print('Target shape:', y_train.shape)
print('\nFirst 5 target values (log transformed):')
print(y_train.head())

Train features shape: (1458, 79)
Test shape: (1459, 79)
Target shape: (1458,)

First 5 target values (log transformed):
0    12.247699
1    12.109016
2    12.317171
3    11.849405
4    12.429220
Name: SalePrice, dtype: float64


In [20]:
none_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 
             'FireplaceQu', 'GarageType', 'GarageFinish',
             'GarageQual', 'GarageCond', 'BsmtQual', 
             'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
             'BsmtFinType2', 'MasVnrType']

for col in none_cols:
    train[col] = train[col].fillna('None')
    test[col]  = test[col].fillna('None')

zero_cols = ['GarageYrBlt', 'GarageCars', 'GarageArea',
             'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF',
             'TotalBsmtSF', 'MasVnrArea', 'BsmtFullBath', 
             'BsmtHalfBath']

for col in zero_cols:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)

lot_medians = train.groupby('Neighborhood')['LotFrontage'].median()
train['LotFrontage'] = train.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
test['LotFrontage'] = test['Neighborhood'].map(lot_medians).where(test['LotFrontage'].isna(), test['LotFrontage'])

In [ ]:
# Electrical → 1 missing, fill with mode from TRAIN
electrical_mode = train['Electrical'].mode()[0]
train['Electrical'] = train['Electrical'].fillna(electrical_mode)
test['Electrical']  = test['Electrical'].fillna(electrical_mode)

# Row 332 and 948 specific fixes 
bsmt_fintype2_mode = train['BsmtFinType2'].mode()[0]
bsmt_exposure_mode = train['BsmtExposure'].mode()[0]

train['BsmtFinType2'] = train['BsmtFinType2'].fillna(bsmt_fintype2_mode)
test['BsmtFinType2']  = test['BsmtFinType2'].fillna(bsmt_fintype2_mode)

train['BsmtExposure'] = train['BsmtExposure'].fillna(bsmt_exposure_mode)
test['BsmtExposure']  = test['BsmtExposure'].fillna(bsmt_exposure_mode)

In [21]:
# Verify: no missing values remaining
train_missing = train.isnull().sum().sum()
test_missing  = test.isnull().sum().sum()

print(f'Train missing values remaining: {train_missing}')
print(f'Test missing values remaining:  {test_missing}')

Train missing values remaining: 1
Test missing values remaining:  12


In [22]:
# find which column still has missing value in train
print("Train missing columns:")
print(train.isnull().sum()[train.isnull().sum() > 0])

print("\nTest missing columns:")
print(test.isnull().sum()[test.isnull().sum() > 0])

Train missing columns:
Electrical    1
dtype: int64

Test missing columns:
MSZoning       4
Utilities      2
Exterior1st    1
Exterior2nd    1
KitchenQual    1
Functional     2
SaleType       1
dtype: int64


In [23]:
# columns that need mode filling
mode_cols = ['Electrical', 'MSZoning', 'Utilities', 
             'Exterior1st', 'Exterior2nd', 'KitchenQual', 
             'Functional', 'SaleType']

for col in mode_cols:
    mode_val = train[col].mode()[0]
    
    train[col] = train[col].fillna(mode_val)
    test[col]  = test[col].fillna(mode_val)

# verify
print(f'Train missing: {train.isnull().sum().sum()}')
print(f'Test missing:  {test.isnull().sum().sum()}')

Train missing: 0
Test missing:  0


In [24]:
# CREATE NEW FEATURES 

# total area of entire house
train['TotalSF'] = (train['TotalBsmtSF'] + 
                    train['1stFlrSF'] + 
                    train['2ndFlrSF'])
test['TotalSF']  = (test['TotalBsmtSF'] + 
                    test['1stFlrSF'] + 
                    test['2ndFlrSF'])

# total bathrooms (half bath counts as 0.5)
train['TotalBaths'] = (train['FullBath'] + 
                       train['BsmtFullBath'] +
                       train['HalfBath'] * 0.5 + 
                       train['BsmtHalfBath'] * 0.5)
test['TotalBaths']  = (test['FullBath'] + 
                       test['BsmtFullBath'] +
                       test['HalfBath'] * 0.5 + 
                       test['BsmtHalfBath'] * 0.5)

# age features — how old when sold
train['HouseAge']      = train['YrSold'] - train['YearBuilt']
train['RenovationAge'] = train['YrSold'] - train['YearRemodAdd']
train['IsRenovated']   = (train['YearRemodAdd'] != train['YearBuilt']).astype(int)

test['HouseAge']       = test['YrSold'] - test['YearBuilt']
test['RenovationAge']  = test['YrSold'] - test['YearRemodAdd']
test['IsRenovated']    = (test['YearRemodAdd'] != test['YearBuilt']).astype(int)

# porch total area
train['TotalPorch'] = (train['OpenPorchSF'] + 
                       train['EnclosedPorch'] +
                       train['3SsnPorch'] + 
                       train['ScreenPorch'])
test['TotalPorch']  = (test['OpenPorchSF'] + 
                       test['EnclosedPorch'] +
                       test['3SsnPorch'] + 
                       test['ScreenPorch'])

# binary features — has or doesn't have
train['HasPool']      = (train['PoolArea'] > 0).astype(int)
train['HasGarage']    = (train['GarageArea'] > 0).astype(int)
train['HasBasement']  = (train['TotalBsmtSF'] > 0).astype(int)
train['HasFireplace'] = (train['Fireplaces'] > 0).astype(int)

test['HasPool']       = (test['PoolArea'] > 0).astype(int)
test['HasGarage']     = (test['GarageArea'] > 0).astype(int)
test['HasBasement']   = (test['TotalBsmtSF'] > 0).astype(int)
test['HasFireplace']  = (test['Fireplaces'] > 0).astype(int)

# quality * condition interaction
train['OverallScore'] = train['OverallQual'] * train['OverallCond']
test['OverallScore']  = test['OverallQual'] * test['OverallCond']

# verify new features added
print('Train shape:', train.shape)
print('Test shape:', test.shape)
print('\nNew features created:')
new_features = ['TotalSF', 'TotalBaths', 'HouseAge', 'RenovationAge',
                'IsRenovated', 'TotalPorch', 'HasPool', 'HasGarage',
                'HasBasement', 'HasFireplace', 'OverallScore']
print(train[new_features].head())

Train shape: (1458, 90)
Test shape: (1459, 90)

New features created:
   TotalSF  TotalBaths  HouseAge  RenovationAge  IsRenovated  TotalPorch  \
0     2566         3.5         5              5            0          61   
1     2524         2.5        31             31            0           0   
2     2706         3.5         7              6            1          42   
3     2473         2.0        91             36            1         307   
4     3343         3.5         8              8            0          84   

   HasPool  HasGarage  HasBasement  HasFireplace  OverallScore  
0        0          1            1             0            35  
1        0          1            1             1            48  
2        0          1            1             1            35  
3        0          1            1             1            35  
4        0          1            1             1            40  


In [25]:
drop_cols = ['1stFlrSF', 'YearBuilt', 'YearRemodAdd']
train = train.drop(drop_cols, axis=1)
test  = test.drop(drop_cols, axis=1)

print('Train shape:', train.shape)
print('Test shape:', test.shape)

# verify dropped columns are gone
dropped = ['1stFlrSF', 'YearBuilt', 'YearRemodAdd']
for col in dropped:
    print(f'{col} in train: {col in train.columns}')

Train shape: (1458, 87)
Test shape: (1459, 87)
1stFlrSF in train: False
YearBuilt in train: False
YearRemodAdd in train: False
